# 03 — Run and Evaluate the Energy Advisor

Runs the full agent against representative questions and evaluates responses.

**Prerequisites:**
1. `01_db_setup.ipynb` — database and sample data loaded
2. `02_rag_setup.ipynb` — vectorstore indexed
3. `.env` file with `OPENAI_API_KEY`

In [ ]:
import sys, os, json
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

### Step 1 — Initialise the agent

In [ ]:
from energy_advisor import EnergyAdvisorAgent

agent = EnergyAdvisorAgent()
print("Agent ready. Tools:", agent.get_agent_tools())

### Step 2 — Helper to extract answer and tools used

In [ ]:
def run(question, context=None):
    """Run the agent and return a structured summary."""
    result = agent.invoke(question, context=context)
    messages = result["messages"]
    answer = messages[-1].content
    tools_used = [
        msg.name
        for msg in messages
        if hasattr(msg, "name") and msg.name and hasattr(msg, "tool_call_id")
    ]
    return {"answer": answer, "tools_used": list(dict.fromkeys(tools_used))}

### Step 3 — Run test scenarios

In [ ]:
CONTEXT = "Location: San Francisco, CA. I have a 5 kW solar panel system and a Tesla Model 3."

test_cases = [
    {
        "id": "ev_charging",
        "question": "When should I charge my electric car tonight to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
    },
    {
        "id": "solar_usage",
        "question": "How can I better use my solar generation throughout the day?",
        "expected_tools": ["get_weather_forecast", "search_energy_tips"],
    },
    {
        "id": "historical_analysis",
        "question": "What were my main energy consumers in the past week, and how can I reduce costs?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices"],
    },
    {
        "id": "savings_estimate",
        "question": "If I shift my HVAC usage to off-peak hours, how much would I save per year?",
        "expected_tools": ["calculate_energy_savings", "get_electricity_prices"],
    },
    {
        "id": "best_practices",
        "question": "What are the best practices for reducing energy consumption for an EV owner with solar?",
        "expected_tools": ["search_energy_tips"],
    },
]

In [ ]:
results = []

for tc in test_cases:
    print(f"\n{'='*60}")
    print(f"[{tc['id']}] {tc['question']}")
    print('='*60)
    try:
        r = run(tc["question"], context=CONTEXT)
        r["id"] = tc["id"]
        r["expected_tools"] = tc["expected_tools"]
        r["tool_match"] = all(t in r["tools_used"] for t in tc["expected_tools"])
        results.append(r)
        print(f"Tools used  : {r['tools_used']}")
        print(f"Tool match  : {'✓' if r['tool_match'] else '✗'}")
        print(f"\nAnswer:\n{r['answer']}")
    except Exception as e:
        print(f"ERROR: {e}")
        results.append({"id": tc["id"], "error": str(e)})

### Step 4 — Evaluation summary

In [ ]:
passed = sum(1 for r in results if r.get("tool_match") is True)
total = len([r for r in results if "error" not in r])
errors = len([r for r in results if "error" in r])

print("\n=== Evaluation Summary ===")
print(f"Total tests : {len(test_cases)}")
print(f"Passed (tool match): {passed}/{total}")
print(f"Errors      : {errors}")

for r in results:
    status = "ERROR" if "error" in r else ("✓" if r.get("tool_match") else "✗")
    print(f"  [{status}] {r['id']}")